In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. Initialize the standard architecture
model = models.densenet121(weights=None)

# --- FIX 1: The Input Layer (Grayscale vs RGB) ---
# The error says the checkpoint has shape [64, 1, 7, 7].
# We must replace the first layer (conv0) to accept 1 input channel instead of 3.
# Standard DenseNet conv0 args: out_channels=64, kernel=7, stride=2, padding=3
model.features.conv0 = nn.Conv2d(in_channels=1, 
                                 out_channels=64, 
                                 kernel_size=7, 
                                 stride=2, 
                                 padding=3, 
                                 bias=False)

# --- FIX 2: The Classifier Layer (Class Count) ---
# The error says the checkpoint has shape [14, 1024].
# We must replace the classifier to output 14 classes.
num_classes_in_pth = 14
in_features = model.classifier.in_features  # 1024
model.classifier = nn.Linear(in_features, num_classes_in_pth)

# 3. Now load the weights
path_to_file = 'best_nih_densenet121.pth'
checkpoint = torch.load(path_to_file, map_location='cpu', weights_only=True)

# Handle cases where checkpoint is a dict or just the weights
state_dict = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint

try:
    model.load_state_dict(state_dict, strict=True)
    print("Success: Model loaded with 1-channel input and 14 classes.")
except RuntimeError as e:
    print(f"Still failing: {e}")

In [ ]:
load_custom_densenet('best_nih_densenet121.pth', num_classes=3, device='cpu')